Current 4.0.1 numerical outputs and current-baseline Figure 3 plotting cells. The reader does not replay 2.6.0 or generate old coefficient masks; figures use saved selected fits, aligned out-of-fold results and existing permutations without refitting.


In [ ]:
# Current4.0.1 numerical reader: select one explicit current candidate.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from cca401_pipeline import load_final
from cca401.preferred import CURRENT
model = 'fc'
volume_policy = "both"
master_seed = 1
outer_folds, inner_folds = 10, 5
branch = "B"
selection_path = CURRENT / "selection" / volume_policy / model.replace("+", "_") / f"seed_{master_seed}_{inner_folds}fold" / "selection.json"
selection = json.loads(selection_path.read_text())
choice = next(c for c in selection["choices"] if c["branch"] == branch)
fit_directory = CURRENT / choice["fit_directory"]
manifest, final_model = load_final(fit_directory)
print("Current full-data-CV-selected parameters:", choice["tau"])


In [ ]:
# Final full-sample fitted coefficients, with no coefficient filtering.
brain_loadings = final_model["brain"]
behavior_loadings = final_model["behavior"]
maps116 = pd.read_csv(fit_directory / "maps116.csv")
symptom_loadings = pd.read_csv(fit_directory / "symptom_loadings.csv", index_col=0)
display(symptom_loadings)
display(maps116)


In [ ]:
# OOF evaluation remains separate from the full-sample refit.
oof_directory = CURRENT / "oof" / volume_policy / model.replace("+", "_") / f"{outer_folds}x{inner_folds}" / f"seed_{master_seed}"
oof_summary = pd.read_csv(oof_directory / "summary.csv")
display(oof_summary[oof_summary.branch == branch])


### Figure 3: current baseline subpanels

Run the current numerical reader above first. These original plotting cells use saved current results; no model is refit.


In [ ]:
# Current-baseline data for the original Figure 3 plotting cells.
from cca401 import preferred
from cca401.nested_utils import SCALES_23
from plots import *
from pathlib import Path
from utils import get_mmpi_labels
from IPython.display import display

ROOT = Path.cwd()
figure_output_dir = ROOT / 'figs' / 'fig3'
scales = SCALES_23
sig_dims = [0, 1]
sig_idx_plot = np.array(sig_dims)
with np.load(preferred.oof_directory(model) / 'B_aligned.npz') as observed:
    U_fc_cv_fix = observed['brain'].copy()
    U_behav_cv_fix = observed['behavior'].copy()
    avg_fc_cv = observed['x_score'].copy()
    avg_behav_cv = observed['y_score'].copy()
U_fc_ref_fix = final_model['brain'].copy()
U_behav_ref_fix = final_model['behavior'].copy()
if model == 'nc':
    with np.load(CURRENT / 'figure_inputs' / 'sdc_display_aligned.npz') as aligned:
        U_fc_cv_fix = aligned['brain_cv'].copy()
        U_behav_cv_fix = aligned['behavior_cv'].copy()
        avg_fc_cv = aligned['x_score'].copy()
        avg_behav_cv = aligned['y_score'].copy()
        U_fc_ref_fix = aligned['brain_ref'].copy()
        U_behav_ref_fix = aligned['behavior_ref'].copy()
permutation_dir = ROOT / 'results' / 'baseline_followups_20261031' / 'permutations'
nulls = pd.read_csv(permutation_dir / 'permutation_nulls.csv')
model_nulls = nulls.loc[nulls['model'] == model].sort_values('permutation_index')
r_perm = model_nulls[['r_1', 'r_2']].to_numpy(float)
inference = pd.read_csv(permutation_dir / 'inference.csv')
r_cv = inference.loc[inference['model'] == model].sort_values('dimension')['observed_r'].to_numpy(float)

def resolve_save_path(key, stem):
    figure_output_dir.mkdir(parents=True, exist_ok=True)
    return str(figure_output_dir / stem)

save_path_brain_topk = resolve_save_path('brain_topk_loadings', 'brain_topk_loadings_fLNM')



In [ ]:
# ===== Brain Top20 Loadings (network-colored, no legend) =====
from pathlib import Path


def load_atlas_labels(atlas_label_txt: str) -> list[str]:
    p = Path(atlas_label_txt)
    if not p.exists():
        raise FileNotFoundError(f'Atlas label file not found: {p}')
    lines = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
    labels = lines[::2]
    return labels


def build_brain_topk_for_plot_single_modality(
    U_brain_ref: np.ndarray,
    U_brain_cv: np.ndarray,
    atlas_labels: list[str],
    dims: list[int],
    topk: int = 20,
):
    if U_brain_ref.ndim != 2:
        raise ValueError(f'U_brain_ref must be 2D, got shape={U_brain_ref.shape}')
    if U_brain_cv.ndim != 3:
        raise ValueError(f'U_brain_cv must be 3D, got shape={U_brain_cv.shape}')

    n_fold, n_feat, n_dims_all = U_brain_cv.shape
    if U_brain_ref.shape[0] != n_feat:
        raise ValueError('Reference and CV feature counts do not match')
    if len(atlas_labels) != n_feat:
        raise ValueError(f'Atlas labels length ({len(atlas_labels)}) must equal brain feature count ({n_feat})')

    dims = [int(d) for d in dims]
    cv_mean_mat = np.zeros((n_feat, len(dims)), dtype=float)

    for j, d in enumerate(dims):
        if d < 0 or d >= n_dims_all:
            raise IndexError(f'dim index out of range: {d} (n_dims={n_dims_all})')
        data = U_brain_cv[:, :, d].copy()
        denom = np.abs(data.mean(axis=0)).max()
        if denom == 0:
            denom = 1.0
        data = data / denom
        cv_mean_mat[:, j] = data.mean(axis=0)

    all_idx = np.arange(n_feat, dtype=int)
    selected_by_dim = []
    for j in range(len(dims)):
        score = np.abs(cv_mean_mat[:, j])
        k = min(int(topk), int(n_feat))
        order = np.argsort(score)[::-1][:k]
        selected_by_dim.append(all_idx[order])

    union_idx = []
    seen = set()
    for arr in selected_by_dim:
        for gi in arr.tolist():
            if gi not in seen:
                seen.add(gi)
                union_idx.append(int(gi))
    if len(union_idx) == 0:
        raise ValueError('No features selected for plotting.')

    union_idx = np.array(union_idx, dtype=int)
    cv_plot = U_brain_cv[:, union_idx][:, :, dims].copy()

    pos_of = {int(g): i for i, g in enumerate(union_idx.tolist())}
    for j, sel in enumerate(selected_by_dim):
        keep = np.zeros(len(union_idx), dtype=bool)
        for g in sel.tolist():
            keep[pos_of[int(g)]] = True
        cv_plot[:, ~keep, j] = 0.0

    variants = [atlas_labels[i] for i in union_idx.tolist()]
    variant_categories = [extract_network_code(v) or 'Unknown' for v in variants]

    rows = []
    for j, d in enumerate(dims):
        for rank, gi in enumerate(selected_by_dim[j].tolist(), start=1):
            net = extract_network_code(atlas_labels[int(gi)])
            rows.append({
                'dim': int(d) + 1,
                'rank_abs': int(rank),
                'feat_idx': int(gi),
                'feat_name': atlas_labels[int(gi)],
                'network': net if net is not None else 'Unknown',
                'cv_mean': float(cv_mean_mat[int(gi), j]),
            })

    top_tbl = pd.DataFrame(rows)
    return cv_plot, variants, variant_categories, top_tbl, selected_by_dim


if len(sig_dims) == 0:
    raise ValueError('sig_dims is empty. Brain plotting requires at least one canonical dimension.')

atlas_path = ROOT / 'atlas' / 'Schaefer2018_100Parcels_7Networks_order_Tian_Subcortex_S1_label_friendly.txt'
atlas_labels = load_atlas_labels(str(atlas_path))

TOPK = 20
dims_plot = [int(d) for d in sig_dims]

cv_plot, variants_plot, variant_categories, brain_top20_table, selected_by_dim = build_brain_topk_for_plot_single_modality(
    U_brain_ref=U_fc_ref_fix,
    U_brain_cv=U_fc_cv_fix,
    atlas_labels=atlas_labels,
    dims=dims_plot,
    topk=TOPK,
)

network_colors = build_brain_network_color_map(PALLETE)
category_colors = dict(network_colors)
if any(c == 'Unknown' for c in variant_categories):
    category_colors['Unknown'] = '#9E9E9E'

ordered = NETWORK_ORDER_BRAIN + (['Unknown'] if 'Unknown' in set(variant_categories) else [])
local_dims = list(range(len(dims_plot)))

plot_brain_loadings(
    cv_plot,
    dims=local_dims,
    title='',
    variants=variants_plot,
    variant_categories=variant_categories,
    category_colors=category_colors,
    category_order=ordered,
    x_label=f'Top{TOPK} Brain Loadings (a.u.)',
    network_palette=PALLETE,
    show_legend=False,
    show_network_legend=True,
    network_legend_loc='upper left',
    network_legend_ncol=1,
    network_legend_title=None,
    network_legend_each_axis=True,
    save_path=save_path_brain_topk,
)

print('brain_top20_table shape:', brain_top20_table.shape)
for dim_val in sorted(brain_top20_table['dim'].unique()):
    print(f'\n--- Top 20 Brain Loadings for Dimension {dim_val} ---')
    dim_table = brain_top20_table[brain_top20_table['dim'] == dim_val]
    display(dim_table.sort_values('rank_abs').head(20))


In [ ]:
behav_variants = get_mmpi_labels(keys=scales)
plot_loadings(
    U_behav_cv_fix,
    variants=behav_variants,
    dims=sig_dims,
    title="",
    legend_fontsize=13,
    legend_loc='auto',
    save_path=resolve_save_path(
        "behav_cv_loadings",
        "behav_cv_loadings_fLNM",
    ),
)


In [ ]:
plot_correlation(
    avg_fc_cv,
    avg_behav_cv,
    dims=sig_dims,
    dot_size=20,
    title=None,
    x_label="FDC",
    save_path=resolve_save_path(
        "brain_behav_corr",
        "plot_correlation_fLNM",
    ),
)


In [ ]:
for color_index, dimension in enumerate(sig_idx_plot):
    fig, ax = plot_permutation_hist(
        r_perm,
        r_cv,
        idx=int(dimension),
        color=MULTI_COLOR[color_index % len(MULTI_COLOR)],
        save_path=resolve_save_path(
            "perm_hist",
            f"perm_hist_fLNM_cd{int(dimension) + 1}",
        ),
    )
    plt.show()


In [ ]:
# Adopted Top 20 brain maps and shared colorbars, from the main plotting module.
plot_brain_maps(model, figure_output_dir)
